# HQCNN for Medical Image Classification

**Hybrid Quantum-Classical CNN - VU MIF PhD DNN Course**

Models: A (linear), B (classical bottleneck), B-linear (capacity-matched), C (8-qubit VQC)

Dataset: DermaMNIST | Primary metric: Macro F1 | Framework: PennyLane + PyTorch

In [ ]:
!pip install pennylane pennylane-lightning medmnist scikit-learn -q

In [ ]:
!git clone https://github.com/Sasan-Ansarian/hqcnn-medical-imaging.git
%cd hqcnn-medical-imaging

In [ ]:
import torch
import pennylane as qml
print(f'PyTorch: {torch.__version__}')
print(f'PennyLane: {qml.__version__}')
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

In [ ]:
from src.models.factory import build_model

model_a = build_model({'name': 'resnet18_small_baseline_frozen'}, 7)
model_b = build_model({'name': 'resnet18_bottleneck_32_8_frozen'}, 7)
model_c = build_model({'name': 'resnet18_quantum_32_8_frozen'}, 7)

def count_params(m): return sum(p.numel() for p in m.parameters() if p.requires_grad)
print(f'Model A trainable params: {count_params(model_a):,}')
print(f'Model B trainable params: {count_params(model_b):,}')
print(f'Model C trainable params: {count_params(model_c):,}')

In [ ]:
import pennylane as qml
import numpy as np

n_qubits, n_layers = 8, 2
dev = qml.device('default.qubit', wires=n_qubits)

@qml.qnode(dev)
def circuit(inputs, weights):
    qml.AngleEmbedding(inputs, wires=range(n_qubits), rotation='Y')
    for layer in range(n_layers):
        for i in range(n_qubits):
            qml.RY(weights[layer, i], wires=i)
        for i in range(n_qubits - 1):
            qml.CNOT(wires=[i, i+1])
    return [qml.expval(qml.PauliZ(i)) for i in range(n_qubits)]

print(qml.draw(circuit)(np.zeros(n_qubits), np.zeros((n_layers, n_qubits))))

## Key Results (full experiments on VU MIF HPC)

| Phase | Config | Classical F1 | Quantum F1 |
|---|---|---|---|
| Frozen baseline | 512->8, 100% data | A: 0.472 | C: 0.317 |
| Best interface | 512->32->8, 100% | B: 0.473 | C: 0.464 |
| End-to-end | 512->32->8, 100% | B: 0.502 | C: 0.471 |
| Transition regime | 8->2, 20% data | B-lin: 0.180 | C acc: 0.414 |

**Finding:** Classical baselines outperform quantum on Macro F1 under all standard conditions.